# GLOBAL LNN SCI v1.4
## Cross-Market Direction Prediction with LNN/CfC

이 노트북은 기존 **KOSPI 200 / 20영업일 / 5-Seed** 연구 코드를 기반으로,
SCI/Q2급 후속 연구를 위한 **다시장(multi-market) 공통 실험 프레임워크**로 개조한 v1입니다.

### 연구 대상
- KOSPI 200
- S&P 500
- NASDAQ-100
- Nikkei 225
- EURO STOXX 50

### v1에 포함된 기능
1. 시장별 설정(config) 기반 데이터 로딩
2. `COMMON / LOCAL / COMMON_LOCAL` 피처 세트
3. 미래정보 누출 방지를 위한 **ffill only** 및 외부변수 lag 설정
4. 20영업일 후 상승/하락 이진분류
5. RNN / LSTM / GRU / Transformer / LTC(LNN) / CfC
6. 5-Seed 반복실험
7. Rolling/Expanding HMM 함수
8. Transfer Learning helper
9. 거래비용 포함 backtest
10. McNemar / Friedman용 결과 테이블 생성

> **중요**  
> 이 파일의 해외시장 auxiliary ticker들은 "실험 파이프라인을 작동시키기 위한 초기 후보"입니다.
> 실제 투고 전에는 각 변수의 경제적 타당성·거래시간·시차를 다시 검증하고,
> 필요한 경우 공식/검증된 CSV 데이터로 교체하는 것을 권장합니다.

### v1.2 핵심 수정
- Loss ablation: `CE / MILD_WCE / MILD_FOCAL`
- Mild class weighting: inverse-frequency가 아니라 **sqrt inverse-frequency**
- Validation set에서 최적 threshold 선택 (`0.30~0.70`)
- Test에는 validation에서 정한 threshold를 그대로 고정 적용
- `Pred_Pos_Rate`(상승 예측 비율) 자동 출력
- Always-Up / Always-Down / Persistence baseline 유지


### v1.3 핵심 변경
- S&P 500 실험을 `COMMON_LOCAL` 피처셋으로 전환
- 기존 COMMON 5개 피처에 `VIX / US10Y / DXY` 추가
- 모델 / seed / loss / threshold protocol은 v1.2와 동일하게 유지
- COMMON-only 대비 COMMON+LOCAL 개선 여부 확인


### v1.4 핵심 변경
- S&P500 local factor를 level-only에서 변화형 피처까지 확장
  - VIX level + 1일 변화율 + 5일 변화율
  - US10Y level + 1일 변화(bp proxy) + 5일 변화
  - DXY log return + 5일 return
- Validation을 단일 2019~2021 구간이 아니라 **walk-forward folds(2019 / 2020 / 2021)** 로 변경
- 각 fold의 validation 확률을 합쳐 최종 threshold를 선택
- Test(2022~2025)는 threshold 선택에 사용하지 않음
- loss / model / seed protocol은 그대로 유지


In [1]:
# ============================================================
# 0. Environment check
# ============================================================
import sys, platform, os, random, copy, gc, math, warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
from hmmlearn.hmm import GaussianHMM
from scipy import stats

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from ncps.torch import LTC, CfC
except Exception as e:
    LTC = CfC = None
    print("⚠️ ncps import failed:", e)

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Platform: Linux-6.18.44-x86_64-with-glibc2.41
Torch: 2.14.1+cpu
DEVICE: cpu


## 1. 공통 실험 설정

기존 논문과의 연결성을 위해 기본값은 그대로 유지합니다.

- Seed: `[42, 52, 62, 72, 82]`
- Lookback: 20 trading days
- Forecast horizon: 20 trading days
- Train: 2010–2018
- Validation: 2019–2021
- Test: 2022–2025
- Threshold: 0.5

In [2]:
# ============================================================
# 1. Global configuration
# ============================================================
SEEDS = [42, 52, 62, 72, 82]

START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

TRAIN_END = "2018-12-31"
VAL_END   = "2021-12-31"
TEST_END  = "2025-12-31"

WINDOW_SIZE = 20
HORIZON = 20

HIDDEN_SIZE = 128
NUM_LAYERS = 1
DROPOUT = 0.20
MAX_LR = 0.005
EPOCHS = 100
PATIENCE = 10
BATCH_SIZE = 64
THRESHOLD = 0.5  # baseline/default only; learned models use validation-optimized threshold

# v1.2 loss ablation
LOSS_MODES = ["CE", "MILD_WCE", "MILD_FOCAL"]
FOCAL_GAMMA = 1.0

# validation threshold search
THRESHOLD_GRID = np.round(np.arange(0.30, 0.701, 0.01), 2)

FEATURE_SETS = ["COMMON", "LOCAL", "COMMON_LOCAL"]

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

print("✅ configuration ready")

✅ configuration ready


## 2. 시장별 설정

`aux`의 각 변수는 `{ticker, lag}` 형식입니다.

- 미국시장은 당일 종가 기준 사용이 가능한 변수는 lag=0
- 한국/일본/유럽에서 미국시장 변수를 쓰는 경우 기본적으로 lag=1로 두어
  **거래시간 차이로 인한 미래정보 누출 위험을 보수적으로 차단**합니다.
- KOSPI 200은 기존 연구 데이터와의 일관성을 위해 로컬 CSV를 우선하도록 설계했습니다.

In [3]:
# ============================================================
# 2. Market configuration
# ============================================================
MARKET_CONFIG = {
    "KOSPI200": {
        "primary_source": "csv",
        "primary_ticker": None,
        "primary_csv": "KOSPI200_daily_2010_2025.csv",
        "aux": {
            "USD_KRW": {"ticker": "KRW=X", "lag": 1},
            "SOX":     {"ticker": "^SOX", "lag": 1},
            "VIX":     {"ticker": "^VIX", "lag": 1},
        },
        # 기존 논문의 Foreign_Net은 아래 custom CSV로 추가 가능
        "custom_csv": {
            # "Foreign_Net": "KOSPI200_foreigner_net.csv",
        },
    },
    "SP500": {
        "primary_source": "yfinance",
        "primary_ticker": "^GSPC",
        "primary_csv": None,
        "aux": {
            "VIX":   {"ticker": "^VIX", "lag": 0},
            "US10Y": {"ticker": "^TNX", "lag": 0},
            "DXY":   {"ticker": "DX-Y.NYB", "lag": 0},
        },
        "custom_csv": {},
    },
    "NASDAQ100": {
        "primary_source": "yfinance",
        "primary_ticker": "^NDX",
        "primary_csv": None,
        "aux": {
            "VXN":   {"ticker": "^VXN", "lag": 0},
            "SOX":   {"ticker": "^SOX", "lag": 0},
            "US10Y": {"ticker": "^TNX", "lag": 0},
        },
        "custom_csv": {},
    },
    "NIKKEI225": {
        "primary_source": "yfinance",
        "primary_ticker": "^N225",
        "primary_csv": None,
        "aux": {
            "USD_JPY": {"ticker": "JPY=X", "lag": 1},
            "VIX":     {"ticker": "^VIX", "lag": 1},
            "SP500":   {"ticker": "^GSPC", "lag": 1},
        },
        "custom_csv": {},
    },
    "EUROSTOXX50": {
        "primary_source": "yfinance",
        "primary_ticker": "^STOXX50E",
        "primary_csv": None,
        "aux": {
            "EUR_USD": {"ticker": "EURUSD=X", "lag": 1},
            "VIX":     {"ticker": "^VIX", "lag": 1},
            "SP500":   {"ticker": "^GSPC", "lag": 1},
        },
        "custom_csv": {},
    },
}

MARKETS = list(MARKET_CONFIG.keys())
print("Markets:", MARKETS)

Markets: ['KOSPI200', 'SP500', 'NASDAQ100', 'NIKKEI225', 'EUROSTOXX50']


## 3. 데이터 로딩 엔진

- `bfill()`은 사용하지 않습니다.
- auxiliary series는 해당 시장 캘린더에 맞춰 `ffill()`만 수행합니다.
- 로컬 CSV는 `Date/날짜` 및 `Close/종가`를 자동 탐색합니다.

In [4]:
# ============================================================
# 3. Data loading helpers
# ============================================================
from pathlib import Path

DATA_DIR = Path(".")  # 필요시 실제 데이터 폴더로 수정

def _detect_date_col(df):
    candidates = [c for c in df.columns if str(c).lower() in ["date", "datetime"] or "날짜" in str(c)]
    if not candidates:
        candidates = [df.columns[0]]
    return candidates[0]

def _detect_value_col(df, preferred=None):
    if preferred and preferred in df.columns:
        return preferred
    for name in ["Close", "Adj Close", "종가", "close", "adjclose"]:
        if name in df.columns:
            return name
    numeric = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric:
        raise ValueError("No numeric value column found.")
    return numeric[0]

def load_csv_series(path, out_name="Close", preferred_col=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path.resolve()}")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    dcol = _detect_date_col(df)
    vcol = _detect_value_col(df, preferred_col)
    df[dcol] = pd.to_datetime(df[dcol])
    s = pd.to_numeric(df[vcol], errors="coerce")
    out = pd.DataFrame({out_name: s.values}, index=df[dcol])
    return out.sort_index()

def download_yf_series(ticker, out_name, start=START_DATE, end=END_DATE):
    if yf is None:
        raise ImportError("yfinance가 없습니다. `pip install yfinance` 후 다시 실행하세요.")
    raw = yf.download(
        ticker, start=start, end=pd.Timestamp(end) + pd.Timedelta(days=1),
        auto_adjust=False, progress=False
    )
    if raw.empty:
        raise RuntimeError(f"No data returned for {ticker}")
    # yfinance 버전별 MultiIndex 대응
    if isinstance(raw.columns, pd.MultiIndex):
        if ("Adj Close", ticker) in raw.columns:
            s = raw[("Adj Close", ticker)]
        elif ("Close", ticker) in raw.columns:
            s = raw[("Close", ticker)]
        else:
            s = raw.xs("Close", axis=1, level=0).iloc[:, 0]
    else:
        col = "Adj Close" if "Adj Close" in raw.columns else "Close"
        s = raw[col]
    return pd.DataFrame({out_name: pd.to_numeric(s, errors="coerce")}).sort_index()

def load_primary_market(market):
    cfg = MARKET_CONFIG[market]
    if cfg["primary_source"] == "csv":
        path = DATA_DIR / cfg["primary_csv"]
        return load_csv_series(path, out_name="Close")
    return download_yf_series(cfg["primary_ticker"], "Close")

def load_custom_feature(path, feature_name):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    dcol = _detect_date_col(df)
    vcol = _detect_value_col(df, feature_name if feature_name in df.columns else None)
    df[dcol] = pd.to_datetime(df[dcol])
    return pd.DataFrame(
        {feature_name: pd.to_numeric(df[vcol], errors="coerce").values},
        index=df[dcol]
    ).sort_index()

def load_market_frame(market):
    cfg = MARKET_CONFIG[market]
    base = load_primary_market(market).copy()

    # yfinance auxiliary factors
    for fname, meta in cfg.get("aux", {}).items():
        aux = download_yf_series(meta["ticker"], fname)
        if int(meta.get("lag", 0)) > 0:
            aux[fname] = aux[fname].shift(int(meta["lag"]))
        base = base.join(aux, how="left")

    # optional custom CSV factors
    for fname, fpath in cfg.get("custom_csv", {}).items():
        custom = load_custom_feature(DATA_DIR / fpath, fname)
        base = base.join(custom, how="left")

    # 절대 bfill 금지
    aux_cols = [c for c in base.columns if c != "Close"]
    if aux_cols:
        base[aux_cols] = base[aux_cols].ffill()

    return base.loc[START_DATE:END_DATE].copy()

## 4. 공통 피처 / 시장 특화 피처

### COMMON
시장마다 동일한 정의를 적용:
- Log_Ret
- Realized Volatility (20d)
- Momentum (20d)
- Disparity (20d MA 대비)
- RSI(14)

### LOCAL
각 시장 config의 auxiliary variables

### COMMON_LOCAL
둘을 모두 사용

이렇게 하면 **Ablation Study**가 자동으로 가능합니다.

In [5]:
# ============================================================
# 4. Feature engineering (v1.4)
# ============================================================
def add_common_features(df):
    x = df.copy()

    # COMMON
    x["Log_Ret"] = np.log(x["Close"]).diff()
    x["Vol20"] = x["Log_Ret"].rolling(20).std() * np.sqrt(252)
    x["Momentum20"] = x["Close"].pct_change(20)

    ma20 = x["Close"].rolling(20).mean()
    x["Disparity20"] = (x["Close"] / ma20) - 1.0

    delta = x["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    x["RSI14"] = 100 - (100 / (1 + rs))

    # TARGET
    x["Forward_Ret20"] = x["Close"].shift(-HORIZON) / x["Close"] - 1.0
    x["Target"] = (x["Forward_Ret20"] > 0).astype(float)
    x.loc[x["Forward_Ret20"].isna(), "Target"] = np.nan

    # --------------------------------------------------------
    # LOCAL TRANSFORMATIONS
    # level + change 형태로 확장
    # --------------------------------------------------------
    if "VIX" in x.columns:
        x["VIX_Change1"] = x["VIX"].pct_change(1)
        x["VIX_Change5"] = x["VIX"].pct_change(5)

    if "US10Y" in x.columns:
        # ^TNX 계열의 level change 자체를 사용
        x["US10Y_Change1"] = x["US10Y"].diff(1)
        x["US10Y_Change5"] = x["US10Y"].diff(5)

    if "DXY" in x.columns:
        x["DXY_LogRet1"] = np.log(x["DXY"]).diff(1)
        x["DXY_LogRet5"] = np.log(x["DXY"]).diff(5)

    return x

COMMON_FEATURES = [
    "Log_Ret", "Vol20", "Momentum20", "Disparity20", "RSI14"
]

LOCAL_FEATURE_MAP = {
    "KOSPI200": [
        "USD_KRW", "SOX", "VIX"
    ],
    "SP500": [
        "VIX", "VIX_Change1", "VIX_Change5",
        "US10Y", "US10Y_Change1", "US10Y_Change5",
        "DXY_LogRet1", "DXY_LogRet5"
    ],
    "NASDAQ100": [
        "VXN", "SOX", "US10Y"
    ],
    "NIKKEI225": [
        "USD_JPY", "VIX", "SP500"
    ],
    "EUROSTOXX50": [
        "EUR_USD", "VIX", "SP500"
    ],
}

def get_feature_columns(df, market, feature_set):
    local_cols = [
        c for c in LOCAL_FEATURE_MAP.get(market, [])
        if c in df.columns
    ]

    if feature_set == "COMMON":
        cols = COMMON_FEATURES
    elif feature_set == "LOCAL":
        cols = local_cols
    elif feature_set == "COMMON_LOCAL":
        cols = COMMON_FEATURES + local_cols
    else:
        raise ValueError(feature_set)

    cols = [c for c in cols if c in df.columns]

    if len(cols) == 0:
        raise ValueError(f"{market}/{feature_set}: feature columns are empty")

    return cols

def build_market_dataset(market):
    raw = load_market_frame(market)
    df = add_common_features(raw)
    return df

def dataset_diagnostics(df, market):
    print(f"\n[{market}] {df.index.min().date()} ~ {df.index.max().date()}")
    print("rows:", len(df))
    print("missing %:")
    print((df.isna().mean().sort_values(ascending=False).head(20) * 100).round(2))

## 5. 시계열 분할 / 표준화 / Tensor 생성

스케일러는 **Train 구간에만 fit**합니다.

또한 `Target(t)`는 `t+20`의 지수 수준을 기준으로 생성하므로,
Train/Validation/Test 경계에서 미래구간을 침범하지 않도록
각 구간 내부에서 시퀀스를 생성합니다.

In [6]:
# ============================================================
# 5. Walk-forward split / scaling / tensors (v1.4)
# ============================================================

WF_FOLDS = [
    # train_end, val_start, val_end
    ("2018-12-31", "2019-01-01", "2019-12-31"),
    ("2019-12-31", "2020-01-01", "2020-12-31"),
    ("2020-12-31", "2021-01-01", "2021-12-31"),
]

def trim_horizon_tail(split_df, horizon=HORIZON):
    if len(split_df) <= horizon:
        raise ValueError(f"Split too short to trim {horizon} rows.")
    return split_df.iloc[:-horizon].copy()

def print_class_distribution(name, y):
    y = pd.Series(y).dropna().astype(int)
    counts = y.value_counts().sort_index()
    total = len(y)
    down = int(counts.get(0, 0))
    up = int(counts.get(1, 0))
    print(
        f"{name:<14} n={total:5d} | "
        f"Down={down:5d} ({down/total:.2%}) | "
        f"Up={up:5d} ({up/total:.2%})"
    )

def make_sequences(X, y, dates, fwd_ret, window=WINDOW_SIZE):
    xs, ys, ds, rs = [], [], [], []

    for i in range(window, len(X)):
        if np.isnan(y[i]):
            continue

        xs.append(X[i-window:i])
        ys.append(int(y[i]))
        ds.append(dates[i])
        rs.append(float(fwd_ret[i]) if np.isfinite(fwd_ret[i]) else np.nan)

    if not xs:
        raise RuntimeError("No sequences created.")

    return (
        torch.tensor(np.asarray(xs), dtype=torch.float32),
        torch.tensor(np.asarray(ys), dtype=torch.long),
        pd.DatetimeIndex(ds),
        np.asarray(rs, dtype=float),
    )

def prepare_split_tensors(train_df, val_df, features):
    train_df = trim_horizon_tail(train_df, HORIZON)
    val_df = trim_horizon_tail(val_df, HORIZON)

    train_df = train_df.dropna(
        subset=features + ["Target", "Forward_Ret20", "Close"]
    )
    val_df = val_df.dropna(
        subset=features + ["Target", "Forward_Ret20", "Close"]
    )

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(train_df[features])
    Xva = scaler.transform(val_df[features])

    tr = make_sequences(
        Xtr,
        train_df["Target"].values,
        train_df.index,
        train_df["Forward_Ret20"].values
    )

    va = make_sequences(
        Xva,
        val_df["Target"].values,
        val_df.index,
        val_df["Forward_Ret20"].values
    )

    return tr, va, scaler, train_df, val_df

def prepare_market_walkforward(df, market, feature_set, verbose=True):
    features = get_feature_columns(df, market, feature_set)

    needed = features + ["Target", "Forward_Ret20", "Close"]
    work = df[needed].replace([np.inf, -np.inf], np.nan).copy()

    fold_bundles = []

    if verbose:
        print("\n[Walk-forward validation folds]")

    for fold_id, (train_end, val_start, val_end) in enumerate(WF_FOLDS, start=1):
        train_df = work.loc[:train_end].copy()
        val_df = work.loc[val_start:val_end].copy()

        tr, va, scaler, tr_df, va_df = prepare_split_tensors(
            train_df, val_df, features
        )

        if verbose:
            print(f"\nFold {fold_id}: Train <= {train_end} | Val {val_start}~{val_end}")
            print_class_distribution("Train", tr_df["Target"])
            print_class_distribution("Validation", va_df["Target"])

        fold_bundles.append({
            "fold_id": fold_id,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "features": features,
            "train": tr,
            "val": va,
            "scaler": scaler,
            "train_df": tr_df,
            "val_df": va_df,
        })

    # Final train for test: 2010~2021
    final_train_df = work.loc[:VAL_END].copy()
    final_test_df = work.loc[
        pd.Timestamp(VAL_END) + pd.Timedelta(days=1):TEST_END
    ].copy()

    final_train_df = trim_horizon_tail(final_train_df, HORIZON)
    final_test_df = trim_horizon_tail(final_test_df, HORIZON)

    final_train_df = final_train_df.dropna(
        subset=features + ["Target", "Forward_Ret20", "Close"]
    )
    final_test_df = final_test_df.dropna(
        subset=features + ["Target", "Forward_Ret20", "Close"]
    )

    final_scaler = StandardScaler()
    Xtr_final = final_scaler.fit_transform(final_train_df[features])
    Xte = final_scaler.transform(final_test_df[features])

    final_train_tensor = make_sequences(
        Xtr_final,
        final_train_df["Target"].values,
        final_train_df.index,
        final_train_df["Forward_Ret20"].values
    )

    test_tensor = make_sequences(
        Xte,
        final_test_df["Target"].values,
        final_test_df.index,
        final_test_df["Forward_Ret20"].values
    )

    if verbose:
        print("\n[Final train/test]")
        print_class_distribution("Final Train", final_train_df["Target"])
        print_class_distribution("Test", final_test_df["Target"])

    return {
        "features": features,
        "folds": fold_bundles,
        "final_train": final_train_tensor,
        "test": test_tensor,
        "final_scaler": final_scaler,
        "raw_df": df,
        "final_train_df": final_train_df,
        "test_df": final_test_df,
    }

## 6. 모델 정의

기존 연구의 RNN/LSTM/GRU/LTC(LNN)/CfC 구조를 유지하면서
SCI 비교군으로 **Transformer baseline**을 추가했습니다.

> `LNN`은 이 코드에서 `ncps.torch.LTC` 기반으로 구현합니다.

In [7]:
# ============================================================
# 6. Models
# ============================================================
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class RecurrentModel(nn.Module):
    def __init__(self, input_size, h, cell="LSTM", n_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        d_p = dropout if n_layers > 1 else 0.0
        if cell == "RNN":
            self.rnn = nn.RNN(input_size, h, num_layers=n_layers, dropout=d_p, batch_first=True)
        elif cell == "LSTM":
            self.rnn = nn.LSTM(input_size, h, num_layers=n_layers, dropout=d_p, batch_first=True)
        elif cell == "GRU":
            self.rnn = nn.GRU(input_size, h, num_layers=n_layers, dropout=d_p, batch_first=True)
        else:
            raise ValueError(cell)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(h, 2)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(self.dropout(out[:, -1, :]))

class LNNModel(nn.Module):
    def __init__(self, input_size, h, dropout=DROPOUT):
        super().__init__()
        if LTC is None:
            raise ImportError("ncps is required for LNN/LTC")
        self.ltc = LTC(input_size, h, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(h, 2)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.ltc(x)
        return self.fc(self.dropout(out[:, -1, :]))

class CfCModel(nn.Module):
    def __init__(self, input_size, h, dropout=DROPOUT):
        super().__init__()
        if CfC is None:
            raise ImportError("ncps is required for CfC")
        self.cfc = CfC(input_size, h, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(h, 2)
        self.apply(init_weights)

    def forward(self, x):
        out, _ = self.cfc(x)
        return self.fc(self.dropout(out[:, -1, :]))

class TransformerClassifier(nn.Module):
    def __init__(self, input_size, h=HIDDEN_SIZE, nhead=4, num_layers=2, dropout=DROPOUT):
        super().__init__()
        self.in_proj = nn.Linear(input_size, h)
        layer = nn.TransformerEncoderLayer(
            d_model=h, nhead=nhead, dim_feedforward=h*2,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(h)
        self.fc = nn.Linear(h, 2)
        self.apply(init_weights)

    def forward(self, x):
        z = self.in_proj(x)
        z = self.encoder(z)
        z = self.norm(z[:, -1, :])
        return self.fc(z)

def build_model(model_name, input_size):
    if model_name in ["RNN", "LSTM", "GRU"]:
        return RecurrentModel(input_size, HIDDEN_SIZE, cell=model_name)
    if model_name == "Transformer":
        return TransformerClassifier(input_size)
    if model_name == "LNN":
        return LNNModel(input_size, HIDDEN_SIZE)
    if model_name == "CfC":
        return CfCModel(input_size, HIDDEN_SIZE)
    raise ValueError(model_name)

MODEL_NAMES = ["RNN", "LSTM", "GRU", "Transformer", "LNN", "CfC"]

In [8]:
# ============================================================
# 7. Loss ablation + walk-forward threshold selection (v1.4)
# ============================================================
def compute_mild_class_weights(y_tensor):
    y = y_tensor.detach().cpu().numpy().astype(int)

    counts = np.bincount(y, minlength=2).astype(float)
    counts[counts == 0] = 1.0

    inv = len(y) / (2.0 * counts)
    w = np.sqrt(inv)
    w = w / w.mean()

    return torch.tensor(w, dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=FOCAL_GAMMA):
        super().__init__()

        if class_weights is not None:
            self.register_buffer(
                "class_weights",
                class_weights.clone().detach()
            )
        else:
            self.class_weights = None

        self.gamma = float(gamma)

    def forward(self, inputs, targets):
        ce = F.cross_entropy(
            inputs,
            targets,
            weight=self.class_weights,
            reduction="none"
        )

        pt = torch.exp(-ce)

        return (((1.0 - pt) ** self.gamma) * ce).mean()

def build_criterion(loss_mode, yt):
    loss_mode = str(loss_mode).upper()

    if loss_mode == "CE":
        return nn.CrossEntropyLoss().to(DEVICE)

    mild_weights = compute_mild_class_weights(yt).to(DEVICE)

    if loss_mode == "MILD_WCE":
        return nn.CrossEntropyLoss(
            weight=mild_weights
        ).to(DEVICE)

    if loss_mode == "MILD_FOCAL":
        return FocalLoss(
            class_weights=mild_weights,
            gamma=FOCAL_GAMMA
        ).to(DEVICE)

    raise ValueError(f"Unknown loss_mode: {loss_mode}")

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience = patience
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False
        self.best_state = None

    def __call__(self, score, model):
        if score > self.best_score:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

def calculate_metrics(y_true, probs=None, threshold=THRESHOLD, preds=None):
    y_true = np.asarray(y_true).astype(int)

    if preds is None:
        probs = np.asarray(probs, dtype=float)
        pred = (probs > threshold).astype(int)
    else:
        pred = np.asarray(preds).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Macro_F1": f1_score(
            y_true, pred, average="macro", zero_division=0
        ),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Pred_Pos_Rate": float(pred.mean()),
    }

def select_validation_threshold(y_true, probs, grid=THRESHOLD_GRID):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)

    records = []

    for t in grid:
        pred = (probs > float(t)).astype(int)

        ba = balanced_accuracy_score(y_true, pred)
        mf1 = f1_score(
            y_true, pred,
            average="macro",
            zero_division=0
        )
        ppr = pred.mean()

        records.append(
            (float(t), ba, mf1, ppr)
        )

    tab = pd.DataFrame(
        records,
        columns=[
            "Threshold",
            "Balanced_Accuracy",
            "Macro_F1",
            "Pred_Pos_Rate"
        ]
    )

    tab["Dist_to_0.5"] = (
        tab["Threshold"] - 0.5
    ).abs()

    best = (
        tab.sort_values(
            [
                "Balanced_Accuracy",
                "Macro_F1",
                "Dist_to_0.5"
            ],
            ascending=[False, False, True]
        )
        .iloc[0]
    )

    return (
        float(best["Threshold"]),
        tab.drop(columns="Dist_to_0.5")
    )

def train_one_fold(
    model,
    Xt, yt,
    Xv, yv,
    seed_for_loader=None,
    loss_mode="CE",
    use_amp=False
):
    """
    Walk-forward fold용.
    validation probability만 반환.
    """
    model = model.to(DEVICE)

    gen = torch.Generator()

    if seed_for_loader is not None:
        gen.manual_seed(int(seed_for_loader))

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=gen,
        pin_memory=(DEVICE.type == "cuda"),
        num_workers=0
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=MAX_LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=MAX_LR,
        steps_per_epoch=max(1, len(loader)),
        epochs=EPOCHS
    )

    criterion = build_criterion(
        loss_mode, yt
    )

    stopping = EarlyStopping(PATIENCE)

    scaler_amp = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    Xv_dev = Xv.to(DEVICE)
    yv_np = yv.numpy()

    for epoch in range(EPOCHS):
        model.train()

        for bx, by in loader:
            bx = bx.to(DEVICE)
            by = by.to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):
                logits = model(bx)
                loss = criterion(logits, by)

            scaler_amp.scale(loss).backward()

            scaler_amp.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            scaler_amp.step(optimizer)
            scaler_amp.update()
            scheduler.step()

        model.eval()

        with torch.no_grad():
            v_probs_epoch = torch.softmax(
                model(Xv_dev),
                dim=1
            )[:, 1].cpu().numpy()

        v_pred_epoch = (
            v_probs_epoch > 0.5
        ).astype(int)

        score = balanced_accuracy_score(
            yv_np, v_pred_epoch
        )

        stopping(score, model)

        if stopping.early_stop:
            break

    if stopping.best_state is not None:
        model.load_state_dict(
            stopping.best_state
        )

    model.eval()

    with torch.no_grad():
        val_probs = torch.softmax(
            model(Xv_dev),
            dim=1
        )[:, 1].cpu().numpy()

    return model.cpu(), val_probs

def train_final_model(
    model,
    Xt, yt,
    Xte,
    seed_for_loader=None,
    loss_mode="CE",
    use_amp=False
):
    """
    Final train(2010~2021) 전체로 재학습 후 test probability 반환.
    별도 validation이 없으므로 fixed epoch rule 사용:
    EPOCHS의 60%만 학습(최소 20 epoch).
    """
    model = model.to(DEVICE)

    gen = torch.Generator()

    if seed_for_loader is not None:
        gen.manual_seed(int(seed_for_loader))

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=gen,
        pin_memory=(DEVICE.type == "cuda"),
        num_workers=0
    )

    final_epochs = max(20, int(EPOCHS * 0.60))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=MAX_LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=MAX_LR,
        steps_per_epoch=max(1, len(loader)),
        epochs=final_epochs
    )

    criterion = build_criterion(
        loss_mode, yt
    )

    scaler_amp = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(final_epochs):
        model.train()

        for bx, by in loader:
            bx = bx.to(DEVICE)
            by = by.to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):
                logits = model(bx)
                loss = criterion(logits, by)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            scaler_amp.step(optimizer)
            scaler_amp.update()
            scheduler.step()

    model.eval()

    Xte_dev = Xte.to(DEVICE)

    with torch.no_grad():
        test_probs = torch.softmax(
            model(Xte_dev),
            dim=1
        )[:, 1].cpu().numpy()

    return model.cpu(), test_probs

def walkforward_threshold_for_model(
    bundle,
    model_name,
    seed,
    loss_mode
):
    """
    2019 / 2020 / 2021 validation fold 확률을 합쳐
    threshold를 한 번만 선택.
    """
    all_val_y = []
    all_val_probs = []

    for fold in bundle["folds"]:
        Xt, yt, _, _ = fold["train"]
        Xv, yv, _, _ = fold["val"]

        set_seed(seed)

        model = build_model(
            model_name,
            Xt.shape[-1]
        )

        model, val_probs = train_one_fold(
            model,
            Xt, yt,
            Xv, yv,
            seed_for_loader=seed,
            loss_mode=loss_mode
        )

        all_val_y.append(
            yv.numpy()
        )
        all_val_probs.append(
            val_probs
        )

        del model
        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    y_val_all = np.concatenate(
        all_val_y
    )

    probs_val_all = np.concatenate(
        all_val_probs
    )

    best_threshold, threshold_table = (
        select_validation_threshold(
            y_val_all,
            probs_val_all,
            THRESHOLD_GRID
        )
    )

    return (
        best_threshold,
        threshold_table,
        y_val_all,
        probs_val_all
    )

## 8. 단일 시장 5-Seed 실행기

먼저 **한 시장**에서 정상 작동을 확인한 뒤,
마지막에 5개 시장 전체 loop를 돌리는 것을 권장합니다.

In [9]:
# ============================================================
# 8. Walk-forward experiment runner (v1.4)
# ============================================================
def persistence_predictions(test_dates, raw_df):
    px = raw_df["Close"].dropna()
    preds = []

    for dt in pd.DatetimeIndex(test_dates):
        if dt not in px.index:
            preds.append(1)
            continue

        loc = px.index.get_loc(dt)

        if isinstance(loc, slice):
            loc = loc.start

        if loc < HORIZON:
            preds.append(1)
        else:
            preds.append(
                int(
                    px.iloc[loc] >
                    px.iloc[loc - HORIZON]
                )
            )

    return np.asarray(
        preds,
        dtype=int
    )

def baseline_rows(
    market,
    feature_set,
    y_true,
    test_dates,
    raw_df
):
    y = np.asarray(
        y_true
    ).astype(int)

    baselines = {
        "Always-Up":
            np.ones_like(y),

        "Always-Down":
            np.zeros_like(y),

        "Persistence":
            persistence_predictions(
                test_dates,
                raw_df
            ),
    }

    rows = []

    for name, pred in baselines.items():
        met = calculate_metrics(
            y,
            preds=pred
        )

        rows.append({
            "Market": market,
            "Feature_Set": feature_set,
            "Model": name,
            "Loss": "BASELINE",
            "Seed": "BASELINE",
            "Threshold": np.nan,
            **met
        })

    return rows

def run_market_experiment(
    market,
    feature_set="COMMON_LOCAL",
    model_names=None,
    seeds=SEEDS,
    loss_modes=LOSS_MODES
):
    if model_names is None:
        model_names = MODEL_NAMES

    df = build_market_dataset(
        market
    )

    bundle = prepare_market_walkforward(
        df,
        market,
        feature_set,
        verbose=True
    )

    Xt_final, yt_final, _, _ = (
        bundle["final_train"]
    )

    Xte, yte, test_dates, fwd_rets = (
        bundle["test"]
    )

    rows = baseline_rows(
        market,
        feature_set,
        yte.numpy(),
        test_dates,
        df
    )

    pred_store = {}

    for loss_mode in loss_modes:
        print(
            f"\n--- LOSS: {loss_mode} ---"
        )

        for seed in seeds:
            for model_name in model_names:

                # --------------------------
                # Step 1. Walk-forward threshold
                # --------------------------
                (
                    best_threshold,
                    threshold_table,
                    val_y_all,
                    val_probs_all
                ) = walkforward_threshold_for_model(
                    bundle,
                    model_name,
                    seed,
                    loss_mode
                )

                # --------------------------
                # Step 2. Final train on 2010~2021
                # --------------------------
                set_seed(seed)

                final_model = build_model(
                    model_name,
                    Xt_final.shape[-1]
                )

                final_model, test_probs = (
                    train_final_model(
                        final_model,
                        Xt_final,
                        yt_final,
                        Xte,
                        seed_for_loader=seed,
                        loss_mode=loss_mode
                    )
                )

                # --------------------------
                # Step 3. Test metrics
                # --------------------------
                met = calculate_metrics(
                    yte.numpy(),
                    probs=test_probs,
                    threshold=best_threshold
                )

                val_met = calculate_metrics(
                    val_y_all,
                    probs=val_probs_all,
                    threshold=best_threshold
                )

                print(
                    f"{model_name:<12} "
                    f"seed={seed:<3} "
                    f"loss={loss_mode:<10} "
                    f"thr={best_threshold:.2f} "
                    f"WF-BA={val_met['Balanced_Accuracy']:.4f} "
                    f"TEST-BA={met['Balanced_Accuracy']:.4f} "
                    f"MacroF1={met['Macro_F1']:.4f} "
                    f"PPR={met['Pred_Pos_Rate']:.3f}"
                )

                rows.append({
                    "Market": market,
                    "Feature_Set": feature_set,
                    "Model": model_name,
                    "Loss": loss_mode,
                    "Seed": seed,
                    "Threshold": best_threshold,
                    "WF_Val_BA":
                        val_met["Balanced_Accuracy"],
                    "WF_Val_Macro_F1":
                        val_met["Macro_F1"],
                    **met
                })

                pred_store[
                    (
                        model_name,
                        loss_mode,
                        seed
                    )
                ] = {
                    "dates":
                        test_dates,

                    "y_true":
                        yte.numpy().copy(),

                    "val_y":
                        val_y_all.copy(),

                    "val_probs":
                        val_probs_all.copy(),

                    "test_probs":
                        test_probs.copy(),

                    "probs":
                        test_probs.copy(),

                    "threshold":
                        float(best_threshold),

                    "threshold_table":
                        threshold_table.copy(),

                    "forward_return":
                        fwd_rets.copy(),

                    "state_dict":
                        copy.deepcopy(
                            final_model.state_dict()
                        ),

                    "input_size":
                        Xt_final.shape[-1],
                }

                del final_model
                gc.collect()

                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()

    result = pd.DataFrame(
        rows
    )

    return (
        result,
        pred_store,
        bundle
    )

def summarize_seed_results(result_df):
    metrics = [
        "Accuracy",
        "Balanced_Accuracy",
        "Recall",
        "Precision",
        "Macro_F1",
        "F1",
        "Pred_Pos_Rate"
    ]

    extra_metrics = [
        "WF_Val_BA",
        "WF_Val_Macro_F1"
    ]

    base = result_df[
        result_df["Seed"].astype(str)
        == "BASELINE"
    ].copy()

    learned = result_df[
        result_df["Seed"].astype(str)
        != "BASELINE"
    ].copy()

    base_cols = [
        "Market",
        "Feature_Set",
        "Model",
        "Loss",
        "Threshold",
    ] + extra_metrics + metrics

    for col in extra_metrics:
        if col not in base.columns:
            base[col] = np.nan

    base_out = base[
        base_cols
    ].copy()

    for col in metrics:
        base_out[col] = (
            base_out[col]
            .map(
                lambda x:
                f"{x:.4f}"
                if pd.notna(x)
                else "-"
            )
        )

    for col in extra_metrics:
        base_out[col] = "-"

    base_out["Threshold"] = "-"

    if len(learned):
        group_cols = [
            "Market",
            "Feature_Set",
            "Model",
            "Loss"
        ]

        cols_for_summary = (
            metrics
            + extra_metrics
            + ["Threshold"]
        )

        mean = learned.groupby(
            group_cols
        )[cols_for_summary].mean()

        std = learned.groupby(
            group_cols
        )[cols_for_summary].std()

        learned_out = mean.copy()

        for col in cols_for_summary:
            learned_out[col] = [
                f"{m:.4f}±{s:.4f}"
                for m, s
                in zip(
                    mean[col].values,
                    std[col].values
                )
            ]

        learned_out = (
            learned_out
            .reset_index()
        )

        learned_out = learned_out[
            [
                "Market",
                "Feature_Set",
                "Model",
                "Loss",
                "Threshold",
            ]
            + extra_metrics
            + metrics
        ]

        return pd.concat(
            [
                base_out,
                learned_out
            ],
            ignore_index=True
        )

    return base_out

## 9. Rolling / Expanding HMM

현재 시점까지의 과거 데이터만 이용해 HMM을 반복 추정합니다.

- `mode="expanding"`: 과거 전체 누적
- `mode="rolling"`: 최근 `lookback_days`만 사용
- 각 test date에서 **해당 날짜 이전 데이터만** fit

In [10]:
# ============================================================
# 9. Rolling / Expanding HMM
# ============================================================
def rolling_hmm_states(
    df,
    test_dates,
    mode="expanding",
    lookback_days=756,       # 약 3년
    refit_every=20,
    n_states=2,
    seeds=SEEDS
):
    hmm_df = df[["Log_Ret", "Vol20"]].dropna().copy()
    test_dates = pd.DatetimeIndex(test_dates)

    states = {}
    cached_model = None
    cached_scaler = None
    cached_order = None

    for i, dt in enumerate(test_dates):
        if (cached_model is None) or (i % refit_every == 0):
            hist = hmm_df[hmm_df.index < dt].copy()
            if mode == "rolling":
                hist = hist.tail(lookback_days)
            if len(hist) < 250:
                states[dt] = np.nan
                continue

            scaler = StandardScaler()
            X = scaler.fit_transform(hist.values.astype(float))

            best_model, best_score = None, -np.inf
            for rs in seeds:
                try:
                    m = GaussianHMM(
                        n_components=n_states,
                        covariance_type="full",
                        n_iter=1000,
                        tol=1e-4,
                        min_covar=1e-4,
                        random_state=rs
                    )
                    m.fit(X)
                    score = m.score(X)
                    if np.isfinite(score) and score > best_score:
                        best_model, best_score = m, score
                except Exception:
                    pass

            if best_model is None:
                states[dt] = np.nan
                continue

            # state 의미를 train history의 volatility 평균으로 정렬
            hist_states = best_model.predict(X)
            state_vol = {}
            for s in range(n_states):
                mask = hist_states == s
                state_vol[s] = hist["Vol20"].values[mask].mean() if mask.any() else np.inf
            cached_order = {
                raw_s: ordered_s
                for ordered_s, raw_s in enumerate(
                    sorted(state_vol, key=state_vol.get)
                )
            }
            cached_model = best_model
            cached_scaler = scaler

        if dt not in hmm_df.index:
            prior = hmm_df[hmm_df.index <= dt]
            if prior.empty:
                states[dt] = np.nan
                continue
            row = prior.iloc[[-1]]
        else:
            row = hmm_df.loc[[dt]]

        x = cached_scaler.transform(row.values.astype(float))
        raw_state = int(cached_model.predict(x)[0])
        states[dt] = int(cached_order[raw_state])

    return pd.Series(states, name="HMM_State").reindex(test_dates)

## 10. Transfer Learning helper

Transfer는 **같은 feature dimension**에서만 직접 가능합니다.
따라서 첫 논문에서는 `COMMON` feature set을 transfer 실험의 기본으로 권장합니다.

비교:
1. Target from scratch
2. Source pretrained → Target fine-tuning

In [11]:
# ============================================================
# 10. Transfer Learning
# ============================================================
# v1.4에서는 walk-forward validation 구조를 먼저 검증하는 단계이므로
# transfer experiment는 잠시 보류합니다.
#
# S&P500 COMMON_LOCAL + walk-forward 결과가 안정화된 뒤
# source/target 공통 피처셋을 기준으로 다시 연결합니다.

def transfer_experiment(*args, **kwargs):
    raise NotImplementedError(
        "v1.4에서는 walk-forward validation 검증을 우선합니다. "
        "Transfer Learning은 다음 단계에서 재연결합니다."
    )

## 11. 거래비용 포함 Backtest

기본 long/cash 전략:
- `prob > threshold` → Long
- 그 외 → Cash
- 거래비용: 0 / 10 / 20 bp

> 20일 horizon 예측과 일별 포지션 갱신은 overlapping signal을 만들 수 있습니다.
> 최종 논문에서는 **월별/20거래일 리밸런싱 버전**도 함께 비교하는 것을 권장합니다.

In [12]:
# ============================================================
# 11. Transaction-cost backtest
# ============================================================
def max_drawdown(equity_curve):
    x = np.asarray(equity_curve, dtype=float)
    peak = np.maximum.accumulate(x)
    dd = x / peak - 1.0
    return np.nanmin(dd)

def backtest_long_cash(
    dates,
    probs,
    forward_returns,
    threshold=0.5,
    cost_bps=10
):
    df = pd.DataFrame({
        "Date": pd.DatetimeIndex(dates),
        "Prob": np.asarray(probs, dtype=float),
        "ForwardRet20": np.asarray(forward_returns, dtype=float),
    }).dropna()

    df["Position"] = (df["Prob"] > threshold).astype(int)
    df["Turnover"] = df["Position"].diff().abs().fillna(df["Position"].abs())
    cost = cost_bps / 10000.0

    # 20일 forward return 기반 단순 경제적 가치 측정
    # 완전한 비중복 포트폴리오 백테스트는 후속 셀에서 리밸런싱 주기를 적용할 것
    df["GrossStrategyRet"] = df["Position"] * df["ForwardRet20"]
    df["NetStrategyRet"] = df["GrossStrategyRet"] - df["Turnover"] * cost
    df["Equity"] = (1.0 + df["NetStrategyRet"]).cumprod()

    r = df["NetStrategyRet"]
    ann_factor = 252 / HORIZON
    ann_ret = (df["Equity"].iloc[-1] ** (ann_factor / max(len(df), 1))) - 1 if len(df) else np.nan
    ann_vol = r.std(ddof=1) * np.sqrt(ann_factor) if len(r) > 1 else np.nan
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol > 0 else np.nan

    summary = {
        "Cost_bps": cost_bps,
        "Final_Equity": df["Equity"].iloc[-1] if len(df) else np.nan,
        "Annualized_Return_proxy": ann_ret,
        "Annualized_Vol_proxy": ann_vol,
        "Sharpe_proxy": sharpe,
        "Max_Drawdown": max_drawdown(df["Equity"]) if len(df) else np.nan,
        "Turnover": df["Turnover"].sum(),
    }
    return summary, df

def backtest_cost_grid(dates, probs, forward_returns, costs=(0, 10, 20)):
    rows = []
    curves = {}
    for c in costs:
        s, d = backtest_long_cash(dates, probs, forward_returns, cost_bps=c)
        rows.append(s)
        curves[c] = d
    return pd.DataFrame(rows), curves

## 12. 통계검정

- 동일 test observations에 대한 두 분류기 비교: McNemar
- 여러 시장에서 모델 rank 비교: Friedman

In [13]:
# ============================================================
# 12. Statistical tests
# ============================================================
def mcnemar_exact(y_true, pred_a, pred_b):
    y = np.asarray(y_true).astype(int)
    a = np.asarray(pred_a).astype(int)
    b = np.asarray(pred_b).astype(int)

    a_correct = a == y
    b_correct = b == y
    n01 = int(np.sum(a_correct & ~b_correct))
    n10 = int(np.sum(~a_correct & b_correct))
    n = n01 + n10

    if n == 0:
        p = 1.0
    else:
        from scipy.stats import binomtest
        p = binomtest(min(n01, n10), n=n, p=0.5, alternative="two-sided").pvalue

    return {
        "only_A_correct": n01,
        "only_B_correct": n10,
        "discordant": n,
        "p_value_exact": p,
    }

def friedman_from_market_table(df, metric="Balanced_Accuracy"):
    # seed 평균 → Market x Model
    pivot = (
        df.groupby(["Market", "Model"])[metric]
          .mean()
          .unstack("Model")
          .dropna(axis=1)
          .dropna(axis=0)
    )
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        raise ValueError("Friedman test needs at least 3 markets and 3 models.")
    arrays = [pivot[c].values for c in pivot.columns]
    stat, p = stats.friedmanchisquare(*arrays)
    return {
        "metric": metric,
        "statistic": stat,
        "p_value": p,
        "market_model_table": pivot,
    }

## 13. 전체 5시장 실행

처음에는 계산량을 줄이기 위해:

```python
RUN_MARKETS = ["SP500"]
RUN_FEATURE_SETS = ["COMMON"]
RUN_MODELS = ["LNN", "LSTM", "Transformer"]
```

정도로 테스트한 뒤 확대하세요.

In [14]:
# ============================================================
# 13. S&P500 COMMON_LOCAL + Walk-forward experiment (v1.4)
# ============================================================

RUN_MARKETS = ["SP500"]
RUN_FEATURE_SETS = ["COMMON_LOCAL"]
RUN_MODELS = [
    "LNN",
    "LSTM",
    "Transformer"
]

RUN_LOSS_MODES = [
    "CE",
    "MILD_WCE",
    "MILD_FOCAL"
]

all_results = []
all_prediction_store = {}
all_bundles = {}

RUN_ALL = True

if RUN_ALL:
    for market in RUN_MARKETS:
        for feature_set in RUN_FEATURE_SETS:

            print(
                "\n"
                + "=" * 70
            )

            print(
                "RUN:",
                market,
                feature_set,
                "+ WALK-FORWARD"
            )

            print(
                "=" * 70
            )

            res, store, bundle = (
                run_market_experiment(
                    market=market,
                    feature_set=feature_set,
                    model_names=RUN_MODELS,
                    seeds=SEEDS,
                    loss_modes=RUN_LOSS_MODES
                )
            )

            all_results.append(
                res
            )

            all_prediction_store[
                (
                    market,
                    feature_set
                )
            ] = store

            all_bundles[
                (
                    market,
                    feature_set
                )
            ] = bundle

    global_results = pd.concat(
        all_results,
        ignore_index=True
    )

    global_summary = (
        summarize_seed_results(
            global_results
        )
    )

    display(
        global_summary
    )

    global_results.to_csv(
        "SCI_v1_4_WALKFORWARD_seed_results.csv",
        index=False,
        encoding="utf-8-sig"
    )

    global_summary.to_csv(
        "SCI_v1_4_WALKFORWARD_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

else:
    print(
        "ℹ️ RUN_ALL=False 입니다."
    )


RUN: SP500 COMMON_LOCAL + WALK-FORWARD



[Walk-forward validation folds]

Fold 1: Train <= 2018-12-31 | Val 2019-01-01~2019-12-31
Train          n= 2224 | Down=  747 (33.59%) | Up= 1477 (66.41%)
Validation     n=  232 | Down=   56 (24.14%) | Up=  176 (75.86%)

Fold 2: Train <= 2019-12-31 | Val 2020-01-01~2020-12-31
Train          n= 2476 | Down=  812 (32.79%) | Up= 1664 (67.21%)
Validation     n=  233 | Down=   62 (26.61%) | Up=  171 (73.39%)

Fold 3: Train <= 2020-12-31 | Val 2021-01-01~2021-12-31
Train          n= 2729 | Down=  874 (32.03%) | Up= 1855 (67.97%)
Validation     n=  232 | Down=   49 (21.12%) | Up=  183 (78.88%)

[Final train/test]
Final Train    n= 2981 | Down=  924 (31.00%) | Up= 2057 (69.00%)
Test           n=  983 | Down=  346 (35.20%) | Up=  637 (64.80%)

--- LOSS: CE ---


LNN          seed=42  loss=CE         thr=0.69 WF-BA=0.5369 TEST-BA=0.4318 MacroF1=0.3786 PPR=0.314


LSTM         seed=42  loss=CE         thr=0.69 WF-BA=0.5844 TEST-BA=0.4299 MacroF1=0.4020 PPR=0.397


Transformer  seed=42  loss=CE         thr=0.52 WF-BA=0.5823 TEST-BA=0.4030 MacroF1=0.4041 PPR=0.575


...

Transformer  seed=72  loss=MILD_FOCAL thr=0.61 WF-BA=0.6293 TEST-BA=0.4408 MacroF1=0.4347 PPR=0.518


LNN          seed=82  loss=MILD_FOCAL thr=0.50 WF-BA=0.5926 TEST-BA=0.4131 MacroF1=0.3561 PPR=0.298


LSTM         seed=82  loss=MILD_FOCAL thr=0.55 WF-BA=0.6508 TEST-BA=0.4340 MacroF1=0.4108 PPR=0.417


Transformer  seed=82  loss=MILD_FOCAL thr=0.50 WF-BA=0.6529 TEST-BA=0.3947 MacroF1=0.3869 PPR=0.479


,Market,Feature_Set,Model,Loss,Threshold,WF_Val_BA,WF_Val_Macro_F1,Accuracy,Balanced_Accuracy,Recall,Precision,Macro_F1,F1,Pred_Pos_Rate
0,SP500,COMMON_LOCAL,Always-Up,BASELINE,-,-,-,0.6604,0.5000,1.0000,0.6604,0.3977,0.7955,1.0000
1,SP500,COMMON_LOCAL,Always-Down,BASELINE,-,-,-,0.3396,0.5000,0.0000,0.0000,0.2535,0.0000,0.0000
2,SP500,COMMON_LOCAL,Persistence,BASELINE,-,-,-,0.5587,0.5151,0.6509,0.6710,0.5147,0.6608,0.6407
3,SP500,COMMON_LOCAL,LNN,CE,0.6380±0.0853,0.5371±0.0279,0.4913±0.0623,0.4123±0.0449,0.4521±0.0213,0.3280±0.0971,0.5947±0.0361,0.4090±0.0395,0.4188±0.0847,0.3605±0.0839
4,SP500,COMMON_LOCAL,LNN,MILD_FOCAL,0.5000±0.0000,0.6037±0.0232,0.5674±0.0290,0.4135±0.0374,0.4413±0.0217,0.3547±0.0770,0.5893±0.0386,0.4098±0.0350,0.4406±0.0717,0.3946±0.0661
5,SP500,COMMON_LOCAL,LNN,MILD_WCE,0.5920±0.0545,0.5395±0.0208,0.4619±0.0565,0.4096±0.0364,0.4476±0.0191,0.3289±0.0758,0.5915±0.0361,0.4073±0.0343,0.4204±0.0706,0.3645±0.0644
6,SP500,COMMON_LOCAL,LSTM,CE,0.6460±0.0297,0.5896±0.0307,0.5405±0.0458,0.4293±0.0276,0.4375±0.0110,0.4119±0.0648,0.5969±0.0189,0.4203±0.0204,0.4859±0.0516,0.4544±0.0584
7,SP500,COMMON_LOCAL,LSTM,MILD_FOCAL,0.5360±0.0114,0.6170±0.0223,0.5625±0.0300,0.4492±0.0220,0.4536±0.0147,0.4399±0.0393,0.6155±0.0173,0.4391±0.0186,0.5127±0.0332,0.4714±0.0306
8,SP500,COMMON_LOCAL,LSTM,MILD_WCE,0.5840±0.0885,0.6039±0.0281,0.5576±0.0390,0.4559±0.0337,0.4438±0.0227,0.4814±0.0598,0.6106±0.0215,0.4370±0.0251,0.5375±0.0441,0.5196±0.0465
9,SP500,COMMON_LOCAL,Transformer,CE,0.5860±0.0817,0.6178±0.0370,0.6094±0.0321,0.4561±0.0633,0.4398±0.0521,0.4906±0.0940,0.6060±0.0464,0.4341±0.0542,0.5406±0.0745,0.5315±0.0659


## 14. 권장 실행 순서

1. **SP500 + COMMON**만 먼저 실행  
2. `LNN / LSTM / Transformer`, seed=42 하나로 smoke test  
3. 5-seed 확대  
4. 5개 시장 확대  
5. `COMMON_LOCAL` 추가  
6. Rolling HMM  
7. Transfer Learning  
8. 0/10/20bp backtest  
9. McNemar + Friedman  
10. 최종 논문 표/그림 자동 생성

### 첫 실행 예시

```python
res, store, bundle = run_market_experiment(
    "SP500",
    feature_set="COMMON",
    model_names=["LNN", "LSTM", "Transformer"],
    seeds=[42]
)
display(res)
```

### 이후 5-seed

```python
res, store, bundle = run_market_experiment(
    "SP500",
    feature_set="COMMON",
    model_names=MODEL_NAMES,
    seeds=SEEDS
)
display(summarize_seed_results(res))
```

---

### 연구 설계상 현재 v1에서 의도적으로 보류한 것

- Dynamic ensemble weighting
- Gating network
- Meta-learning / Multi-task learning
- Domain adaptation
- Contrastive pretraining
- Online / continual learning
- SHAP / integrated gradients
- Attention-LNN hybrid
- Multi-horizon
- Multi-asset portfolio optimization

즉, 이번 SCI 논문의 핵심은 **Cross-market generalization + Regime robustness + Transferability + Economic value**에 집중합니다.

## v1.2 결과 해석 체크포인트

이번 버전에서는 S&P 500에서 다음을 우선 확인합니다.

1. `Persistence`의 Balanced Accuracy(기존 약 0.515)를 learned model이 넘는가?
2. `Pred_Pos_Rate`가 0 또는 1에 지나치게 몰리지 않는가?
3. `CE / MILD_WCE / MILD_FOCAL` 중 어느 loss가 가장 안정적인가?
4. Validation에서 선택된 threshold가 seed별로 지나치게 흔들리지 않는가?
5. LNN이 최소한 Balanced Accuracy / Macro F1에서 baseline보다 일관되게 우수한가?

### 계산량 줄이는 smoke test
먼저 1 seed만 확인하려면:

```python
res, store, bundle = run_market_experiment(
    "SP500",
    feature_set="COMMON",
    model_names=["LNN", "LSTM", "Transformer"],
    seeds=[42],
    loss_modes=["CE", "MILD_WCE", "MILD_FOCAL"]
)
display(summarize_seed_results(res))
```

정상 작동하면 `seeds=SEEDS`로 5-seed를 실행하세요.

## v1.3 S&P500 COMMON_LOCAL 확인

이번 단계에서는 기존 COMMON 피처에 S&P500 특화 변수 `VIX`, `US10Y`, `DXY`를 추가합니다.

다른 설정은 그대로 유지합니다:
- Models: LNN / LSTM / Transformer
- Seeds: 42 / 52 / 62 / 72 / 82
- Loss: CE / MILD_WCE / MILD_FOCAL
- Validation threshold search: 0.30~0.70
- Baselines: Always-Up / Always-Down / Persistence

이번 실험의 핵심은 **COMMON-only 대비 COMMON+LOCAL 개선 여부**입니다.

## v1.4 해석 포인트

이번 버전의 핵심은 하이퍼파라미터 튜닝이 아니라 **검증 구조 개선**입니다.

- 2019 validation fold
- 2020 validation fold
- 2021 validation fold

세 fold의 validation prediction을 합친 뒤 threshold를 하나만 고릅니다.
Test(2022~2025)는 threshold 선택에 전혀 사용하지 않습니다.

또한 S&P500 local feature는 단순 level만 넣지 않고 변화형 피처를 추가합니다.

- VIX / VIX_Change1 / VIX_Change5
- US10Y / US10Y_Change1 / US10Y_Change5
- DXY_LogRet1 / DXY_LogRet5

결과에서 우선 볼 항목:

1. `WF_Val_BA`
2. `Balanced_Accuracy`(Test)
3. `Macro_F1`
4. `Pred_Pos_Rate`
5. threshold seed 안정성
6. Persistence BA(약 0.515) 초과 여부

## 실행 결과 내보내기 및 Persistence 비교
원본 실험의 모델·seed·loss·epoch 설정을 유지합니다. 아래 표의 차이는 Test Balanced Accuracy 기준이며 통계적 유의성을 의미하지 않습니다.

In [15]:

from pathlib import Path
from itertools import product
output_dir = Path('artifacts')
output_dir.mkdir(exist_ok=True)
learned = global_results[global_results['Seed'].astype(str) != 'BASELINE'].copy()
baseline = global_results[global_results['Seed'].astype(str) == 'BASELINE'].copy()
expected = set(product(RUN_MARKETS, RUN_FEATURE_SETS, RUN_MODELS, RUN_LOSS_MODES, SEEDS))
keys = ['Market', 'Feature_Set', 'Model', 'Loss', 'Seed']
assert set(learned[keys].itertuples(index=False, name=None)) == expected
assert len(learned) == len(expected), 'Missing or duplicate runs'
metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1', 'F1', 'Recall', 'Precision',
           'Pred_Pos_Rate', 'WF_Val_BA', 'WF_Val_Macro_F1', 'Threshold']
assert np.isfinite(learned[metrics].to_numpy(dtype=float)).all()
summary = learned.groupby(['Market', 'Feature_Set', 'Model', 'Loss'])[metrics].agg(['mean', 'std'])
summary.columns = ['_'.join(c) for c in summary.columns]
summary = summary.reset_index()
persistence = baseline[baseline['Model'] == 'Persistence'][
    ['Market', 'Feature_Set', 'Balanced_Accuracy', 'Macro_F1', 'Accuracy']
].rename(columns={m: 'Persistence_' + m for m in ['Balanced_Accuracy', 'Macro_F1', 'Accuracy']})
comparison = learned.merge(persistence, on=['Market', 'Feature_Set'], validate='many_to_one')
for metric in ['Balanced_Accuracy', 'Macro_F1', 'Accuracy']:
    comparison['Delta_' + metric] = comparison[metric] - comparison['Persistence_' + metric]
summary = summary.merge(persistence, on=['Market', 'Feature_Set'], validate='many_to_one')
summary['Delta_Balanced_Accuracy_mean'] = summary['Balanced_Accuracy_mean'] - summary['Persistence_Balanced_Accuracy']
summary['Seeds_Beating_Persistence'] = [
    int(((comparison['Market'] == row.Market) & (comparison['Feature_Set'] == row.Feature_Set)
         & (comparison['Model'] == row.Model) & (comparison['Loss'] == row.Loss)
         & (comparison['Delta_Balanced_Accuracy'] > 0)).sum())
    for row in summary.itertuples()
]
global_results.to_csv(output_dir / 'seed_results.csv', index=False, encoding='utf-8-sig')
global_summary.to_csv(output_dir / 'formatted_summary.csv', index=False, encoding='utf-8-sig')
summary.to_csv(output_dir / 'final_results.csv', index=False, encoding='utf-8-sig')
comparison.to_csv(output_dir / 'persistence_comparison.csv', index=False, encoding='utf-8-sig')
baseline.to_csv(output_dir / 'baselines.csv', index=False, encoding='utf-8-sig')
display(summary)
display(baseline)
report = '# S&P500 COMMON_LOCAL + walk-forward results\n\n'
report += summary.to_string(index=False) + '\n\nPersistence and other baselines:\n'
report += baseline.to_string(index=False)
report += '\n\nChanges: appended reporting/export cells only; original experiment settings unchanged.\n'
(output_dir / 'results_report.md').write_text(report, encoding='utf-8')
print('Verified', len(learned), 'learned runs; reports exported to', output_dir.resolve())


,Market,Feature_Set,Model,Loss,Accuracy_mean,Accuracy_std,Balanced_Accuracy_mean,Balanced_Accuracy_std,Macro_F1_mean,Macro_F1_std,...,WF_Val_BA_std,WF_Val_Macro_F1_mean,WF_Val_Macro_F1_std,Threshold_mean,Threshold_std,Persistence_Balanced_Accuracy,Persistence_Macro_F1,Persistence_Accuracy,Delta_Balanced_Accuracy_mean,Seeds_Beating_Persistence
0,SP500,COMMON_LOCAL,LNN,CE,0.412253,0.044887,0.452067,0.021289,0.408978,0.039541,...,0.027872,0.491257,0.062256,0.638,0.085264,0.515074,0.514657,0.558671,-0.063007,0
1,SP500,COMMON_LOCAL,LNN,MILD_FOCAL,0.413499,0.037444,0.441273,0.021742,0.409770,0.034984,...,0.023217,0.567438,0.029012,0.500,0.000000,0.515074,0.514657,0.558671,-0.073801,0
2,SP500,COMMON_LOCAL,LNN,MILD_WCE,0.409553,0.036357,0.447646,0.019094,0.407318,0.034275,...,0.020830,0.461915,0.056546,0.592,0.054498,0.515074,0.514657,0.558671,-0.067428,0
3,SP500,COMMON_LOCAL,LSTM,CE,0.429283,0.027568,0.437473,0.010966,0.420261,0.020439,...,0.030708,0.540464,0.045809,0.646,0.029665,0.515074,0.514657,0.558671,-0.077601,0
4,SP500,COMMON_LOCAL,LSTM,MILD_FOCAL,0.449221,0.022011,0.453608,0.014671,0.439141,0.018581,...,0.022303,0.562499,0.029961,0.536,0.011402,0.515074,0.514657,0.558671,-0.061466,0
5,SP500,COMMON_LOCAL,LSTM,MILD_WCE,0.455867,0.033697,0.443781,0.022746,0.436983,0.025099,...,0.028126,0.557595,0.038970,0.584,0.088487,0.515074,0.514657,0.558671,-0.071293,0
6,SP500,COMMON_LOCAL,Transformer,CE,0.456075,0.063299,0.439778,0.052137,0.434146,0.054181,...,0.036998,0.609430,0.032114,0.586,0.081731,0.515074,0.514657,0.558671,-0.075296,1
7,SP500,COMMON_LOCAL,Transformer,MILD_FOCAL,0.459190,0.048403,0.436937,0.038119,0.433651,0.039673,...,0.019216,0.595453,0.018929,0.540,0.048477,0.515074,0.514657,0.558671,-0.078137,0
8,SP500,COMMON_LOCAL,Transformer,MILD_WCE,0.417445,0.067569,0.403550,0.021916,0.388084,0.034714,...,0.027883,0.588841,0.037923,0.584,0.087636,0.515074,0.514657,0.558671,-0.111524,0


,Market,Feature_Set,Model,Loss,Seed,Threshold,Accuracy,Balanced_Accuracy,Recall,Precision,Macro_F1,F1,Pred_Pos_Rate,WF_Val_BA,WF_Val_Macro_F1
0,SP500,COMMON_LOCAL,Always-Up,BASELINE,BASELINE,NaN,0.660436,0.500000,1.000000,0.660436,0.397749,0.795497,1.000000,NaN,NaN
1,SP500,COMMON_LOCAL,Always-Down,BASELINE,BASELINE,NaN,0.339564,0.500000,0.000000,0.000000,0.253488,0.000000,0.000000,NaN,NaN
2,SP500,COMMON_LOCAL,Persistence,BASELINE,BASELINE,NaN,0.558671,0.515074,0.650943,0.670989,0.514657,0.660814,0.640706,NaN,NaN


Verified 45 learned runs; reports exported to /workspace/global-lnn-sci/artifacts
